# Let an agent pick its own data source with Firecrawl Alexandria

Most research agents treat the web as a pile of pages: search, scrape, and hope the answer survives the HTML. Many questions have a better source. SEC filings, package registries, labor statistics, and earthquake catalogues all publish structured data, and an agent that can find and call those sources gets exact fields, real identifiers, and a source URL on every record.

This notebook builds that agent with the OpenAI Agents SDK and [Firecrawl Alexandria](https://docs.firecrawl.dev), a catalogue of data providers that return typed, sourced records through published contracts. The agent:

1. Searches the catalogue for a provider that fits the question.
2. Reads the provider's contract to learn its inputs and output fields.
3. Calls the provider and gets typed records instead of scraped pages.
4. Falls back to Firecrawl web search and scrape when no provider covers the question.
5. Returns its answer as Structured Outputs, with a source URL and a verbatim quote on every claim.

The main example compares the Risk Factors sections of two companies' latest annual reports (Form 10-K), pulled straight from SEC EDGAR. Then the notebook stores everything the agent fetched in an OpenAI vector store so follow-up questions stay grounded, and runs the same agent on other providers and questions.

**What you need**

- An OpenAI API key in `OPENAI_API_KEY`.
- A Firecrawl API key in `FIRECRAWL_API_KEY`. You can create one at [firecrawl.dev](https://www.firecrawl.dev).
- Python 3.10 or newer.

**Cost.** Catalogue search and contract reads are free. Each provider call costs a few Firecrawl credits (the contract lists the price), and a full run of this notebook uses well under 100 credits. OpenAI usage is a handful of agent runs on `gpt-6.1-sol` plus a small vector store, which the last cell deletes.

## 1. Set up the environment

Install the OpenAI SDK, the Agents SDK, and the Firecrawl Python SDK.

In [ ]:
%pip install --quiet -U "openai>=3.39.0" "openai-agents>=0.24.0" "firecrawl-py>=4.54.0" "pydantic>=2.11"

Note: you may need to restart the kernel to use updated packages.


Both keys come from the environment. `OPENAI_MODEL` defaults to `gpt-6.1-sol`, which balances reasoning quality and cost for multi-step tool use; set it to another Responses API model to compare.

In [ ]:
import json
import os
import re
import time
from datetime import datetime, timezone
from typing import Any, Literal

from agents import Agent, FileSearchTool, ModelSettings, Runner, function_tool
from agents.items import ToolCallItem
from firecrawl import Firecrawl
from firecrawl.v2.utils.error_handler import FirecrawlError
from openai import OpenAI
from openai.types.shared import Reasoning
from pydantic import BaseModel, Field

for name in ("OPENAI_API_KEY", "FIRECRAWL_API_KEY"):
    if not os.getenv(name):
        raise RuntimeError(f"Set {name} before running this notebook.")

MODEL = os.getenv("OPENAI_MODEL", "gpt-6.1-sol")

openai_client = OpenAI()
firecrawl = Firecrawl(api_key=os.environ["FIRECRAWL_API_KEY"])

print(f"Model: {MODEL}")

Model: gpt-6.1-sol


## 2. A short explainer: what is a 10-K?

A **Form 10-K** is the annual report that US public companies file with the Securities and Exchange Commission (SEC). It is longer and more standardized than the glossy annual report sent to shareholders, and every filing follows the same numbered items:

| Item | Section | What it covers |
|---|---|---|
| 1 | Business | What the company does, its segments, and its markets |
| 1A | Risk Factors | The material risks management believes could hurt the business |
| 7 | Management's Discussion and Analysis (MD&A) | Management's explanation of results, liquidity, and trends |
| 8 | Financial Statements | Audited balance sheet, income statement, and cash flows |

**Item 1A, Risk Factors**, is a useful place to compare companies. It is written in plain language, it changes from year to year as new risks appear (tariffs, AI, cyber incidents, weight-loss drugs), and two competitors' lists show where they see the same threats and where they differ.

Every filing lives on **EDGAR**, the SEC's public filing system, under an *accession number* such as `0001628280-26-010047`. A 10-K document can run to several hundred thousand characters of inline XBRL, so scraping the page and asking a model to find Item 1A is slow and fragile. A provider that already splits the filing into items returns exactly the section you asked for, with the filing date, the period it covers, and the EDGAR URL.

## 3. Look under the hood: Alexandria by hand

Before giving the agent any tools, run the three operations it will use yourself. None of these cells calls a model.

**Search the catalogue.** `search` with the `alexandria` source returns provider capabilities that match a plain-language description of the data. Catalogue search is free.

In [ ]:
catalogue = firecrawl.search(
    "Risk Factors section of a company's latest 10-K annual report from SEC EDGAR",
    sources=["alexandria"],
    limit=5,
)

for tool in catalogue.tools:
    print(f"{tool.provider:>10} | {tool.capability:<22} | {tool.description[:90]}...")

   sec-gov | filings/sections       | Item sections of a 10-K or 10-Q as plain text: Item 1 Business, 1A Risk Factors, 7 MD&A, 8...
   sec-gov | filings/filing         | One filing by accession number: its submission metadata (form, dates, 8-K items) and every...
   sec-gov | filings/filings        | List a filer's EDGAR filings newest first, filtered by form type (10-K, 10-Q, 8-K, S-1, 13...
   sec-gov | filings/holdings_13f   | Form 13F-HR holdings of an institutional investment manager: every position from the XML i...
   sec-gov | filings/company        | Resolve a ticker, CIK or company name to its EDGAR filer profile: CIK, name, tickers, exch...
 fiscal-ai | filings/list           | Call to find out what exists and when it was filed, and to get the document links. The fil...
   sec-gov | filings/companies_get  | A single filer's profile or its complete filing history page by page....


**Read the contract.** `find_tools` with a provider and capability returns the full contract: input options with their types and defaults, and the response fields. Contract reads are free too.

In [ ]:
contract_page = firecrawl.find_tools(
    providers=["sec-gov"],
    capabilities=["filings/sections"],
    expand=["options", "response"],
)
contract = contract_page.items[0]

print(contract["description"], "\n")
print("Inputs:")
for option in contract["options"]:
    default = f" (default {option['default']})" if "default" in option else ""
    print(f"  {option['name']}: {option['type']}{default}. {option.get('about', '')}")
print("\nResponse fields:", [field["name"] for field in contract["response"]["fields"]])

Item sections of a 10-K or 10-Q as plain text: Item 1 Business, 1A Risk Factors, 7 MD&A, 8 Financial Statements and so on (10-Q items carry their Part). Defaults to the filer's latest 10-K; pass form=10-Q or an accession number for another report, and items to keep only some sections. 

Inputs:
  accession_number: string. Pattern: ^[0-9]{10}-[0-9]{2}-[0-9]{6}$.
  cik: number. SEC Central Index Key, with or without leading zeros.
  form: string (default 10-K). form
  items: string[]. Item ids to return, e.g. ["1A", "7"]. Default: all detected.
  max_chars: number (default 60000). Per-section text cap.
  name: string. Filer name; the best EDGAR typeahead match is used.
  ticker: string. Exchange ticker as EDGAR lists it (BRK-B or BRK.B both work). Pattern: ^[A-Za-z0-9.\-]{1,12}$.

Response fields: ['company', 'count', 'detected_items', 'document_chars', 'document_url', 'filing', 'observed_at_ms', 'sections', 'source_url']


**Call the provider.** `scrape_alexandria` executes one or more capabilities (up to ten per request) and returns one result per call. Here it fetches Item 1A from one company's latest 10-K. The record carries the filing metadata and the EDGAR document URL, which becomes the source for every claim built on it.

In [ ]:
demo = firecrawl.scrape_alexandria(
    {
        "provider": "sec-gov",
        "capability": "filings/sections",
        "options": {"ticker": "KO", "items": ["1A"]},
    }
)
record = demo.alexandria[0].data
section = record["sections"][0]

print("Company:     ", record["company"]["name"])
print("Form:        ", record["filing"]["form"], "for the period ending", record["filing"]["report_date"])
print("Filed:       ", record["filing"]["filing_date"], "accession", record["filing"]["accession_number"])
print("Source:      ", record["document_url"])
print("Section:     ", section["item"], section["title"], f"({section['char_count']:,} characters)")
print("Credits used:", demo.credits_cost)
print()
print(section["text"][:600], "...")

Company:      COCA COLA CO
Form:         10-K for the period ending 2025-12-31
Filed:        2026-02-20 accession 0001628280-26-010047
Source:       https://www.sec.gov/Archives/edgar/data/21344/000162828026010047/ko-20251231.htm
Section:      1A RISK FACTORS (92,229 characters)
Credits used: 5

In addition to the other information set forth in this report, you should carefully consider the following factors, which could materially affect our business, financial condition and results of operations in future periods. The risks described below are not the only risks facing our Company. Additional risks not currently known to us or that we currently deem to be immaterial also may materially adversely affect our business, financial condition or results of operations in future periods.
RISKS RELATED TO OUR OPERATIONS
Unfavorable general economic and geopolitical conditions could negatively ...


That is the whole pattern: describe the data, read the contract, call the provider. The rest of the notebook hands those three steps to an agent.

## 4. Turn the operations into agent tools

The agent gets five function tools:

| Tool | Firecrawl call | Purpose |
|---|---|---|
| `find_providers` | `search(sources=["alexandria"])` | Find capabilities that match the data the question needs |
| `read_contract` | `find_tools(providers, capabilities)` | Learn a capability's inputs and output fields |
| `call_provider` | `scrape_alexandria` | Fetch typed records |
| `web_search` | `search(sources=["web"])` | Fallback when no provider fits |
| `scrape_page` | `scrape(formats=["markdown"])` | Read one page from the fallback search |

Every tool that fetches content also records it in an `EVIDENCE` store keyed by source URL, and returns that URL to the model in a `sources` list. The model must cite one of those URLs for each claim. Later cells use the same store to check citations and to fill the vector store.

The notebook calls the public Firecrawl Python SDK directly rather than connecting the hosted Firecrawl MCP server. Both expose the same catalogue, but a 10-K section can exceed 60,000 characters, and the notebook needs the full text in Python to verify quotes and to index it. Section 9 shows how to connect the MCP server instead.

In [ ]:
EVIDENCE: dict[str, dict[str, Any]] = {}
MAX_TOOL_OUTPUT_CHARS = 120_000
PROVIDER_RETRIES = 3


def remember(url: str, title: str, text: str, origin: str) -> str:
    """Store fetched content under its source URL and return the URL."""
    EVIDENCE[url] = {
        "title": title,
        "text": text,
        "origin": origin,
        "fetched_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    return url


def record_documents(provider: str, capability: str, data: Any) -> list[str]:
    """Split one provider result into citable documents and return their URLs."""
    if not isinstance(data, dict):
        return []
    origin = f"alexandria:{provider}/{capability}"
    if data.get("sections") and data.get("document_url"):
        company = data.get("company", {}).get("name", "")
        filing = data.get("filing", {})
        urls = []
        for section in data["sections"]:
            url = f"{data['document_url']}#item-{section['item']}"
            title = (
                f"{company} {filing.get('form', '')} for the period ending "
                f"{filing.get('report_date', '')}, Item {section['item']} {section.get('canonical_title', '')}"
            )
            urls.append(remember(url, title, section["text"], origin))
        return urls
    url = data.get("source_url") or next(iter(data.get("source_urls") or []), None)
    if not url:
        return []
    return [remember(url, f"{provider} {capability}", json.dumps(data), origin)]


def truncate(payload: dict[str, Any]) -> str:
    text = json.dumps(payload, default=str)
    if len(text) <= MAX_TOOL_OUTPUT_CHARS:
        return text
    return text[:MAX_TOOL_OUTPUT_CHARS] + '..." [truncated]'

A provider call can occasionally return HTTP 503 with a message that no provider was executed (for example, when a price quote is briefly unavailable). Nothing ran and nothing was charged, so the helper retries it with a short backoff.

In [ ]:
def scrape_alexandria_with_retry(call: dict[str, Any]):
    for attempt in range(1, PROVIDER_RETRIES + 1):
        try:
            return firecrawl.scrape_alexandria(call)
        except FirecrawlError as error:
            not_executed = "No provider was executed" in str(error)
            if getattr(error, "status_code", None) != 503 or not not_executed or attempt == PROVIDER_RETRIES:
                raise
            time.sleep(2 * attempt)


@function_tool
def find_providers(data_needed: str) -> str:
    """Search the Alexandria catalogue for provider capabilities that return the data you need.

    Args:
        data_needed: Plain-language description of the data, for example
            "Risk Factors from a company's latest 10-K".
    """
    result = firecrawl.search(data_needed, sources=["alexandria"], limit=8)
    tools = [
        {"provider": t.provider, "capability": t.capability, "description": t.description}
        for t in (result.tools or [])
    ]
    return json.dumps({"capabilities": tools})


@function_tool
def read_contract(provider: str, capability: str) -> str:
    """Read the full contract (inputs, defaults, and response fields) of one capability.

    Args:
        provider: The provider id from find_providers, for example "sec-gov".
        capability: The capability id from find_providers, for example "filings/sections".
    """
    page = firecrawl.find_tools(providers=[provider], capabilities=[capability], expand=["options", "response"])
    if not page.items:
        return json.dumps({"error": f"No contract found for {provider} {capability}."})
    item = page.items[0]
    return json.dumps(
        {
            "provider": provider,
            "capability": capability,
            "description": item.get("description"),
            "options": item.get("options"),
            "requiresOneOf": item.get("requiresOneOf"),
            "response_fields": [f["name"] for f in item.get("response", {}).get("fields", [])],
        }
    )


@function_tool
def call_provider(provider: str, capability: str, options_json: str) -> str:
    """Execute one Alexandria capability and return its typed records.

    Args:
        provider: The provider id, for example "sec-gov".
        capability: The capability id, for example "filings/sections".
        options_json: A JSON object with the contract inputs, for example
            '{"ticker": "KO", "items": ["1A"]}'. Pass only inputs the contract defines.
    """
    try:
        options = json.loads(options_json or "{}")
    except json.JSONDecodeError as error:
        return json.dumps({"error": f"options_json is not valid JSON: {error}"})
    call = {"provider": provider, "capability": capability, "options": options}
    try:
        response = scrape_alexandria_with_retry(call)
    except FirecrawlError as error:
        return json.dumps({"error": str(error)})
    result = response.alexandria[0]
    if result.error:
        return json.dumps({"error": result.error.message, "code": result.error.code})
    sources = record_documents(provider, capability, result.data)
    return truncate({"sources": sources, "records": result.records, "data": result.data})


@function_tool
def web_search(query: str) -> str:
    """Search the web. Use this only when no Alexandria provider fits the question.

    Args:
        query: The search query.
    """
    result = firecrawl.search(query, sources=["web"], limit=5)
    hits = [
        {"url": hit.url, "title": hit.title, "description": (hit.description or "")[:300]}
        for hit in (result.web or [])
    ]
    return json.dumps({"results": hits})


@function_tool
def scrape_page(url: str) -> str:
    """Fetch one web page as markdown. Use it on a result from web_search.

    Args:
        url: The page URL.
    """
    document = firecrawl.scrape(url, formats=["markdown"], only_main_content=True)
    title = (document.metadata.title if document.metadata else None) or url
    source = remember(url, title, document.markdown or "", "web:scrape")
    return truncate({"sources": [source], "title": title, "markdown": document.markdown})


TOOLS = [find_providers, read_contract, call_provider, web_search, scrape_page]

## 5. Define the Structured Outputs

Every claim in the answer is a `SourcedClaim`: the statement, the exact source URL from a tool result, and a short verbatim quote from that source. Because the quote has to appear in text the agent actually fetched, a validator can check each claim mechanically.

The 10-K comparison has its own schema: the filings it used, the themes both companies flag, and the risks only one of them calls out.

In [ ]:
class SourcedClaim(BaseModel):
    claim: str = Field(description="One factual statement.")
    source_url: str = Field(description="Copied exactly from the sources list of a tool result.")
    quote: str = Field(description="A verbatim excerpt of at most 40 words from that source that supports the claim.")


class FilingUsed(BaseModel):
    company: str
    ticker: str
    form: str
    period_ending: str
    filing_date: str
    accession_number: str
    source_url: str


class SharedTheme(BaseModel):
    theme: str = Field(description="A short name for a risk both companies describe.")
    comparison: str = Field(description="How the two companies frame this risk, and how they differ.")
    evidence: list[SourcedClaim] = Field(description="At least one claim per company.")


class DistinctiveRisk(BaseModel):
    ticker: str
    risk: str
    evidence: SourcedClaim


class RiskFactorComparison(BaseModel):
    filings: list[FilingUsed]
    shared_themes: list[SharedTheme]
    distinctive_risks: list[DistinctiveRisk]
    summary: str = Field(description="Two or three sentences that only restate the sourced findings above.")


class SourcedAnswer(BaseModel):
    answer: str
    route: Literal["alexandria", "web"] = Field(description="alexandria if a provider supplied the data, web otherwise.")
    capabilities_used: list[str] = Field(description="provider/capability pairs that were called, empty for web.")
    claims: list[SourcedClaim]

## 6. Build the agent and compare two 10-Ks

The instructions encode the routing order: catalogue first, contract second, provider third, web last. They also forbid outside knowledge, so every fact in the output has to come from a tool result.

In [ ]:
INSTRUCTIONS = """You answer questions only with data you fetch through your tools.

Route every question in this order:
1. Call find_providers with a description of the data the question needs.
2. If a capability fits, call read_contract for it, then call_provider with only the inputs the contract defines.
   Batch similar entities by calling call_provider once per entity.
3. If no capability fits, or the provider returns an error you cannot fix by changing inputs,
   call web_search and then scrape_page on the most authoritative result.

Citations:
- Every claim needs a source_url copied exactly from the "sources" list of a tool result.
- Every claim needs a quote copied verbatim (at most 40 words) from that source's text.
- Return a claim for every fact stated in the answer.
- Do not use outside knowledge. If the fetched data does not answer part of the question, say so in the answer.
"""


def build_agent(output_type: type[BaseModel]) -> Agent:
    return Agent(
        name="Data source router",
        model=MODEL,
        instructions=INSTRUCTIONS,
        tools=TOOLS,
        output_type=output_type,
        model_settings=ModelSettings(reasoning=Reasoning(effort="medium")),
    )


def print_route(result) -> None:
    """Print the tool calls the agent made, in order."""
    for item in result.new_items:
        if isinstance(item, ToolCallItem):
            name = getattr(item.raw_item, "name", "")
            arguments = getattr(item.raw_item, "arguments", "")
            print(f"  {name}({arguments[:140]})")

In [ ]:
COMPARISON_QUESTION = (
    "Compare the Risk Factors (Item 1A) in the latest 10-K filings of The Coca-Cola Company (KO) "
    "and PepsiCo (PEP). Identify four to six risk themes both companies describe and two or three "
    "risks that only one of them calls out."
)

comparison_agent = build_agent(RiskFactorComparison)
comparison_run = await Runner.run(comparison_agent, COMPARISON_QUESTION, max_turns=20)
comparison: RiskFactorComparison = comparison_run.final_output

print("Tool calls:")
print_route(comparison_run)

Tool calls:
  find_providers({"data_needed":"Latest 10-K filing metadata and full Item 1A Risk Factors for The Coca-Cola Company (KO) and PepsiCo (PEP), to compare share)
  read_contract({"provider":"sec-gov","capability":"filings/sections"})
  call_provider({"provider":"sec-gov","capability":"filings/sections","options_json":"{\"ticker\":\"KO\",\"items\":[\"1A\"],\"max_chars\":160000}"})
  call_provider({"provider":"sec-gov","capability":"filings/sections","options_json":"{\"ticker\":\"PEP\",\"items\":[\"1A\"],\"max_chars\":160000}"})


In [ ]:
for filing in comparison.filings:
    print(f"{filing.ticker}: {filing.form} for the period ending {filing.period_ending}, filed {filing.filing_date}")
    print(f"    {filing.source_url}")

print("\nShared themes")
for theme in comparison.shared_themes:
    print(f"\n- {theme.theme}: {theme.comparison}")
    for claim in theme.evidence:
        print(f'    "{claim.quote}"')
        print(f"      {claim.source_url}")

print("\nDistinctive risks")
for risk in comparison.distinctive_risks:
    print(f"\n- {risk.ticker}: {risk.risk}")
    print(f'    "{risk.evidence.quote}"')
    print(f"      {risk.evidence.source_url}")

print("\nSummary:", comparison.summary)

KO: 10-K for the period ending 2025-12-31, filed 2026-02-20
    https://www.sec.gov/Archives/edgar/data/21344/000162828026010047/ko-20251231.htm#item-1A
PEP: 10-K for the period ending 2025-12-27, filed 2026-02-03
    https://www.sec.gov/Archives/edgar/data/77476/000007747626000007/pep-20251227.htm#item-1A

Shared themes

- Health concerns and changing consumption: Both identify weight-loss drugs as a potential influence on consumption. Coca-Cola frames this within concerns about obesity and sweetened beverages; PepsiCo explicitly names GLP-1 medications within broader changes in diet and consumer preferences.
    "the effects or perceived effects of the usage of weight-loss drugs on consumption patterns"
      https://www.sec.gov/Archives/edgar/data/21344/000162828026010047/ko-20251231.htm#item-1A
    "diet (whether due to changes in consumer behavior and eating habits, increasing use of weight-loss drugs, such as GLP-1 medications, or other factors)"
      https://www.sec.gov/Archive

## 7. Check every claim against what was fetched

Structured Outputs guarantees the shape of the answer, not its truth. The validator below checks two things for every claim:

1. **The source exists.** The `source_url` is one the agent actually fetched in this session.
2. **The quote is real.** After normalizing whitespace and typographic quotes, the quote appears in the fetched text.

A failure does not have to mean a hallucination (a model can trim a quote awkwardly), but it flags exactly which claims a human should read before trusting them.

In [ ]:
def normalize(text: str) -> str:
    text = text.replace("\u2019", "'").replace("\u2018", "'").replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2014", "-").replace("\u2013", "-")
    return re.sub(r"\s+", " ", text).strip().lower()


def check_claim(claim: SourcedClaim) -> tuple[bool, str]:
    evidence = EVIDENCE.get(claim.source_url)
    if evidence is None:
        return False, "source was not fetched in this session"
    quote = normalize(claim.quote).strip(' ."')
    if quote and quote in normalize(evidence["text"]):
        return True, "quote found"
    return False, "quote not found in the fetched text"


def comparison_claims(result: RiskFactorComparison) -> list[SourcedClaim]:
    claims = [claim for theme in result.shared_themes for claim in theme.evidence]
    return claims + [risk.evidence for risk in result.distinctive_risks]


def report(claims: list[SourcedClaim]) -> None:
    passed = 0
    for claim in claims:
        ok, reason = check_claim(claim)
        passed += ok
        print(f"{'PASS' if ok else 'FAIL'}  {claim.claim[:90]}")
        if not ok:
            print(f"      {reason}: {claim.source_url}")
    print(f"\n{passed} of {len(claims)} claims verified against fetched sources.")


report(comparison_claims(comparison))

PASS  KO identifies the actual or perceived effects of weight-loss drugs on consumption as part 
PASS  PEP explicitly identifies GLP-1 medications as a factor that could change consumers' diets
PASS  KO warns that supply-chain disruption and increases in raw-material, packaging, energy and
PASS  KO says higher energy demand can raise energy costs for both the company and independent b
PASS  PEP warns that manufacturing or supply-chain disruption and increased input costs can adve
PASS  KO warns that building e-commerce relationships could damage relationships with key physic
PASS  PEP warns about maintaining relationships across e-commerce, hard-discount and traditional
PASS  KO warns that increasing water scarcity and deteriorating water quality can raise costs, c
PASS  PEP identifies interruption, cessation or relocation of its own or business partners' faci
PASS  KO says cybercriminals increasingly use advanced capabilities, including zero-day vulnerab
PASS  PEP warns that generativ

## 8. Keep follow-up questions grounded with a vector store

The agent fetched two long sections, but it only quoted a few sentences. Follow-up questions ("What does each company say about water?") should be answered from the same filings, not from the model's memory and not by fetching them again.

Upload everything in `EVIDENCE` to an OpenAI vector store. File Search splits each file into chunks of about 800 tokens and returns chunks on their own, so a single header at the top of the file is not enough. `with_source_markers` repeats a `[Source: ...]` line every 1,500 characters, which puts the source URL inside every chunk the model sees. Each file also carries the URL as a file attribute for filtering.

In [ ]:
SOURCE_MARKER_EVERY_CHARS = 1_500


def with_source_markers(url: str, title: str, text: str) -> str:
    """Prefix roughly every 1,500 characters of text with its source, on line boundaries."""
    blocks, current = [], ""
    for line in text.splitlines(keepends=True):
        if current and len(current) + len(line) > SOURCE_MARKER_EVERY_CHARS:
            blocks.append(current)
            current = ""
        current += line
    blocks.append(current)
    return "".join(f"[Source: {url}]\n{block}" for block in blocks)


vector_store = openai_client.vector_stores.create(name="firecrawl-alexandria-evidence")
uploaded_file_ids = []

for url, evidence in EVIDENCE.items():
    header = f"Title: {evidence['title']}\nFetched: {evidence['fetched_at']}\n\n"
    body = header + with_source_markers(url, evidence["title"], evidence["text"])
    file_name = re.sub(r"[^A-Za-z0-9]+", "-", evidence["title"]).strip("-")[:80] + ".txt"
    uploaded = openai_client.files.create(file=(file_name, body.encode("utf-8")), purpose="assistants")
    openai_client.vector_stores.files.create_and_poll(
        vector_store_id=vector_store.id,
        file_id=uploaded.id,
        attributes={"source_url": url, "origin": evidence["origin"]},
    )
    uploaded_file_ids.append(uploaded.id)
    print(f"Indexed {file_name} ({len(evidence['text']):,} characters)")

print("\nVector store:", vector_store.id)

Indexed COCA-COLA-CO-10-K-for-the-period-ending-2025-12-31-Item-1A-Risk-Factors.txt (92,229 characters)


Indexed PEPSICO-INC-10-K-for-the-period-ending-2025-12-27-Item-1A-Risk-Factors.txt (66,107 characters)

Vector store: vs_6ae174c6fd0481918ecfdbb7e1b66aaf


The follow-up agent has a single tool, File Search over that vector store. It answers with the same `SourcedAnswer` schema, citing the `[Source: ...]` URL from each retrieved chunk, so the validator from section 7 still applies.

In [ ]:
FOLLOW_UP_INSTRUCTIONS = """Answer only from the filings in the file search results.
Every retrieved chunk contains a "[Source: URL]" line. Use that URL, copied exactly, as the source_url of every claim,
and quote the supporting text verbatim (at most 40 words).
Return at least one claim for each company you mention. Write the answer as plain prose with no citation
markers; the claims carry the sources. Set route to "alexandria" and list no capabilities.
If the filings do not cover the question, say so."""

follow_up_agent = Agent(
    name="Grounded follow-up",
    model=MODEL,
    instructions=FOLLOW_UP_INSTRUCTIONS,
    tools=[FileSearchTool(vector_store_ids=[vector_store.id], max_num_results=8)],
    output_type=SourcedAnswer,
    model_settings=ModelSettings(reasoning=Reasoning(effort="low")),
)

FOLLOW_UPS = [
    "What does each company say about water availability or water scarcity?",
    "How does each company describe risks from artificial intelligence?",
]

for question in FOLLOW_UPS:
    follow_up = (await Runner.run(follow_up_agent, question)).final_output
    print(f"Q: {question}\nA: {follow_up.answer}\n")
    report(follow_up.claims)
    print("-" * 80)

Q: What does each company say about water availability or water scarcity?
A: Coca-Cola says water is essential to its beverages, agricultural ingredients and manufacturing. Rising demand, overexploitation, pollution, poor management, inadequate access and climate change threaten water availability and quality. Scarcity could increase costs, constrain capacity and damage its reputation and profitability. Climate-related extreme weather could also limit water available for bottling.

PepsiCo says water is essential to manufacturing and producing its raw materials. Insufficient acceptable-quality water, along with pressure to conserve and replenish supplies, could disrupt its supply chain, raise compliance, capital and production costs, and force facilities to interrupt operations, close or relocate. It also identifies risks of missing water-use goals and reputational damage.

Both frame water scarcity as an operational and financial risk. Coca-Cola emphasizes costs and capacity constrain

Q: How does each company describe risks from artificial intelligence?
A: PepsiCo describes AI risk across several areas: more sophisticated cyberattacks and malware; infringement claims arising from unauthorized use of third-party technology or content; and evolving regulation that could increase compliance costs or restrict AI use. It also identifies a strategic risk: failing to make appropriate AI and data-analytics investments to understand consumer trends could weaken demand and its competitive position.

Coca-Cola discusses AI as an amplifier of cybersecurity threats, noting that cybercriminals rapidly integrate generative AI into their capabilities. The surrounding disclosure warns that cybersecurity incidents could disrupt operations, expose confidential or personal data, and cause financial, legal and reputational harm.

In short, the retrieved disclosures frame PepsiCo’s AI exposure more broadly across cybersecurity, intellectual property, regulation and investment decisions, 

## 9. The same pattern on other providers and questions

Nothing in the agent is specific to SEC filings. The catalogue covers companies, people, jobs, finance and filings, public records and government spending, real estate, package registries, developer data, news, research, and more. The same agent, with the generic `SourcedAnswer` schema, picks a different provider for each question below. The last question has no matching provider, so the agent falls back to web search and scrape.

In [ ]:
OTHER_QUESTIONS = [
    "How many GitHub stars does the openai/openai-agents-python repository have, and what is its latest release?",
    "What is the latest US national unemployment rate from the Bureau of Labor Statistics, and for which month?",
    "List the three strongest earthquakes worldwide in the last 30 days with their magnitude and location.",
    "What are three headline features in the official 'What's New in Python 3.14' documentation?",
]

general_agent = build_agent(SourcedAnswer)
other_results = []

for question in OTHER_QUESTIONS:
    run = await Runner.run(general_agent, question, max_turns=15)
    answer: SourcedAnswer = run.final_output
    other_results.append((question, answer))
    print(f"Q: {question}")
    print_route(run)
    print(f"Route: {answer.route} {answer.capabilities_used}")
    print(f"A: {answer.answer}\n")

Q: How many GitHub stars does the openai/openai-agents-python repository have, and what is its latest release?
  find_providers({"data_needed":"Current GitHub star count and latest release for repository openai/openai-agents-python"})
  read_contract({"provider":"github-com","capability":"repositories/repo"})
  read_contract({"provider":"github-com","capability":"repositories/latest_release"})
  call_provider({"provider":"github-com","capability":"repositories/repo","options_json":"{\"repo\":\"openai/openai-agents-python\"}"})
  call_provider({"provider":"github-com","capability":"repositories/latest_release","options_json":"{\"repo\":\"openai/openai-agents-python\"}"})
Route: alexandria ['github-com/repositories/repo', 'github-com/repositories/latest_release']
A: openai/openai-agents-python has **30,344 GitHub stars**. Its latest release is **v0.24.0**, published on **October 15, 2026**.



Q: What is the latest US national unemployment rate from the Bureau of Labor Statistics, and for which month?
  find_providers({"data_needed":"Latest US national unemployment rate from the Bureau of Labor Statistics and its reference month, seasonally adjusted headli)
  read_contract({"provider":"bls-gov","capability":"economic-statistics/bls_unemployment_rate"})
  call_provider({"provider":"bls-gov","capability":"economic-statistics/bls_unemployment_rate","options_json":"{\"seasonally_adjusted\":true}"})
Route: alexandria ['bls-gov/economic-statistics/bls_unemployment_rate']
A: The latest U.S. national unemployment rate reported by the Bureau of Labor Statistics is **4.2% for September 2026**, seasonally adjusted.



Q: List the three strongest earthquakes worldwide in the last 30 days with their magnitude and location.
  find_providers({"data_needed":"The three strongest earthquakes worldwide in the last 30 days, ranked by magnitude, including magnitude, location, and event)
  read_contract({"provider":"earthquake-usgs-gov","capability":"earthquakes/search"})
  call_provider({"provider":"earthquake-usgs-gov","capability":"earthquakes/search","options_json":"{\"event_type\":\"earthquake\",\"order_by\":\"magnitude\)
Route: alexandria ['earthquake-usgs-gov/earthquakes/search']
A: The three strongest earthquakes worldwide in the last 30 days, ranked by magnitude in the USGS results, are:

| Rank | Magnitude | Location | Date (UTC) |
|---|---|---|---|
| 1 | 7.7 | 12 km WSW of Pitaloza Arriba, Panama | October 9, 2026 |
| 2 | 7.0 | 191 km ENE of Nagqu, China | October 22, 2026 |
| 3 | 6.9 | 197 km WNW of Port-Olry, Vanuatu | October 24, 2026 |



Q: What are three headline features in the official 'What's New in Python 3.14' documentation?
  find_providers({"data_needed":"Official Python documentation What's New in Python 3.14 headline new features"})
  web_search({"query":"site.docs.python.org/3.14/whatsnew/3.14.html What's New Python 3.14 summary release highlights"})
  scrape_page({"url":"https://docs.python.org/3/whatsnew/3.14.html"})
Route: web []
A: Three headline features in the official [What’s New in Python 3.14](https://docs.python.org/3/whatsnew/3.14.html) documentation are:

1. **Template string literals (t-strings)** for custom string processing.
2. **Deferred evaluation of annotations**.
3. **Multiple interpreters (subinterpreters) supported in the standard library**.



In [ ]:
print(f"{'Route':<11} {'Capabilities':<48} {'Verified':<9} Question")
for question, answer in other_results:
    verified = sum(check_claim(claim)[0] for claim in answer.claims)
    capabilities = ", ".join(answer.capabilities_used) or "web search + scrape"
    print(f"{answer.route:<11} {capabilities[:47]:<48} {verified}/{len(answer.claims):<7} {question[:60]}")

Route       Capabilities                                     Verified  Question
alexandria  github-com/repositories/repo, github-com/reposi  2/2       How many GitHub stars does the openai/openai-agents-python r
alexandria  bls-gov/economic-statistics/bls_unemployment_ra  1/1       What is the latest US national unemployment rate from the Bu
alexandria  earthquake-usgs-gov/earthquakes/search           3/3       List the three strongest earthquakes worldwide in the last 3
web         web search + scrape                              3/3       What are three headline features in the official 'What's New


### Connecting the hosted MCP server instead

If you would rather not write tool wrappers, the hosted Firecrawl MCP server exposes the same three operations as `firecrawl_search` (with the `alexandria` source), `firecrawl_find_tools`, and `firecrawl_scrape` (with an `alexandria` body), and the Agents SDK can connect to it directly:

```python
from agents.mcp import MCPServerStreamableHttp, create_static_tool_filter

async with MCPServerStreamableHttp(
    name="firecrawl",
    params={
        "url": "https://mcp.firecrawl.dev/v2/mcp",
        "headers": {"Authorization": f"Bearer {os.environ['FIRECRAWL_API_KEY']}"},
    },
    tool_filter=create_static_tool_filter(
        allowed_tool_names=["firecrawl_search", "firecrawl_find_tools", "firecrawl_scrape"]
    ),
) as firecrawl_mcp:
    agent = Agent(name="Data source router", model=MODEL, mcp_servers=[firecrawl_mcp])
    result = await Runner.run(agent, "Compare the Risk Factors in KO's and PEP's latest 10-Ks.")
```

The server keeps large results (such as two full Risk Factors sections) in a temporary workspace and lets the agent read slices of them, which keeps the context small. The tradeoff is that your code never holds the full text, so the citation checks and vector store in this notebook would need an extra step to fetch it.

## 10. Clean up

Delete the vector store and the uploaded files so they do not count against your storage.

In [ ]:
openai_client.vector_stores.delete(vector_store.id)
for file_id in uploaded_file_ids:
    openai_client.files.delete(file_id)
print(f"Deleted vector store {vector_store.id} and {len(uploaded_file_ids)} files.")

Deleted vector store vs_6ae174c6fd0481918ecfdbb7e1b66aaf and 2 files.


## Next steps

- **Swap the question.** Change `COMPARISON_QUESTION` to any two tickers, or ask for Item 7 (MD&A) instead of Item 1A. The same provider returns any item from a 10-K or 10-Q.
- **Pin what you rely on.** A call may include a `version` to pin a published provider workflow; omitting it uses the latest version.
- **Handle terms.** Some providers require accepting terms before first use. When a response reports a terms requirement, it includes a Firecrawl dashboard URL where you can accept them.
- **Watch for changes.** Store the accession number from `filings` and rerun the comparison when a company files a new 10-K.

Firecrawl's documentation covers the [search](https://docs.firecrawl.dev/features/search) and [scrape](https://docs.firecrawl.dev/features/scrape) endpoints, and the Agents SDK documentation covers [tools](https://openai.github.io/openai-agents-python/tools/) and [Structured Outputs](https://openai.github.io/openai-agents-python/agents/#output-types).